<a href="https://colab.research.google.com/github/VainaviS/EndoNet/blob/main/notebooks/yolov8v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile

zip_path = list(uploaded.keys())[0]
extract_path = "/content/glenda"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import json, os

json_path = "/content/glenda/Glenda_v1.5_classes/coco.json"

with open(json_path) as f:
    data = json.load(f)

print("Keys:", data.keys())

# Map image_id → image info
images = {img["id"]: img for img in data["images"]}

cat_map = {cat["id"]: i for i, cat in enumerate(data["categories"])}

print("Category mapping:", cat_map)

In [ ]:
label_dir = "/content/glenda/labels"
os.makedirs(label_dir, exist_ok=True)

for ann in data["annotations"]:
    img_id = ann["image_id"]
    img_info = images[img_id]

    file_name = os.path.splitext(img_info["file_name"])[0]
    width = img_info["width"]
    height = img_info["height"]

    # COCO bbox
    x, y, w, h = ann["bbox"]

    # Convert to YOLO format
    x_center = (x + w/2) / width
    y_center = (y + h/2) / height
    w /= width
    h /= height

    class_id = cat_map[ann["category_id"]]

    label_path = os.path.join(label_dir, file_name + ".txt")

    with open(label_path, "a") as f:
        f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

print("Conversion done!")

In [ ]:
import shutil

src = "/content/glenda/Glenda_v1.5_classes/frames"
dst = "/content/glenda/images"

os.makedirs(dst, exist_ok=True)

for file in os.listdir(src):
    shutil.copy(os.path.join(src, file), os.path.join(dst, file))

print("Images moved!")

In [ ]:
files = os.listdir("/content/glenda/labels")
print("Total label files:", len(files))

# Check one file
with open("/content/glenda/labels/" + files[0]) as f:
    print(f.read())

In [ ]:
import random

img_dir = "/content/glenda/images"
lbl_dir = "/content/glenda/labels"

images = os.listdir(img_dir)
random.shuffle(images)

split = int(0.8 * len(images))

for split_name, split_list in zip(["train", "val"], [images[:split], images[split:]]):
    os.makedirs(f"/content/glenda/images/{split_name}", exist_ok=True)
    os.makedirs(f"/content/glenda/labels/{split_name}", exist_ok=True)

    for img in split_list:
        base = os.path.splitext(img)[0]
        lbl = base + ".txt"

        if os.path.exists(f"{lbl_dir}/{lbl}"):
            shutil.copy(f"{img_dir}/{img}", f"/content/glenda/images/{split_name}/{img}")
            shutil.copy(f"{lbl_dir}/{lbl}", f"/content/glenda/labels/{split_name}/{lbl}")

In [ ]:
import yaml

data_yaml = {
    'path': '/content/glenda',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'peritoneum',
        1: 'ovary',
        2: 'tie',
        3: 'uterus'
    }
}

with open('/content/glenda/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

In [ ]:
!ls /content/glenda/labels/train | wc -l
!ls /content/glenda/images/train | wc -l

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8l.pt')

model.train(
    data='/content/glenda/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    augment=True,
    patience=30,
    project="glenda_bbox",
    name="lesion",
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0
)

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
model.predict(
    source='/content/glenda/images/val',
    save=True,
    conf=0.25
)

In [ ]:
from ultralytics import YOLO

model = YOLO('runs/detect/glenda_bbox/lesion/weights/best.pt')

results = model.predict(
    source='/content/glenda/images/val',
    conf=0.25,
    save=False,
    stream=True
)

In [ ]:
import os
from collections import defaultdict

label_dir = "/content/glenda/labels/val"
image_dir = "/content/glenda/images/val"

class_names = {
    0: 'Peritoneum',
    1: 'Ovary',
    2: 'TIE',
    3: 'Uterus'
}

selected_images = {}

for lbl_file in os.listdir(label_dir):
    lbl_path = os.path.join(label_dir, lbl_file)

    with open(lbl_path) as f:
        classes = set(int(line.split()[0]) for line in f if line.strip())

    for cls in classes:
        if cls not in selected_images:
            img_name = lbl_file.replace(".txt", ".jpg")
            img_path = os.path.join(image_dir, img_name)

            if os.path.exists(img_path):
                selected_images[cls] = img_path

    if len(selected_images) == 4:
        break

print(selected_images)

In [ ]:
images_list = list(selected_images.values())

results = model.predict(
    source=images_list,
    conf=0.25,
    save=False
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for i, (cls, img_path) in enumerate(selected_images.items()):
    r = results[i]
    img = r.plot()  # YOLO draws boxes

    ax = axes[i//2][i%2]
    ax.imshow(img)
    ax.set_title(class_names[cls])
    ax.axis('off')

plt.tight_layout()
plt.savefig("yolo_2x2_grid.png", dpi=300)
plt.show()

In [ ]:
from google.colab import files
files.download("/content/runs/detect/glenda_bbox/lesion/results.png")